# Capstone, a scored tool suite

**Scenario:** a bid management assistant on an ad exchange. Fourteen tools, added by three teams over
two years. A trader says "LI-4471 is spending too fast, take its bids down 20 percent" and the
assistant calls three different write tools with 0.8 in each.

That is **three hands on the same dial**, and none of them knows about the other two. This capstone
turns the two earlier lessons into a number. You do not argue about a tool suite. You score it.

## Mechanics

An eval is a table, not an opinion. One row per task, and the verdict is computed.

| Field | Meaning |
|---|---|
| `task` | What a trader actually typed |
| `tool` | The one tool wired to the exchange for that task |
| `field` | The argument that carries the decision |
| `value` | What that argument has to be for the task to be done |
| `calls` | Everything the model asked for, in order |

A row passes when exactly one call came back, its name is `tool`, and its `field` equals `value`.
Nothing softer, because a suite that half works is one nobody can reason about.

## The picture

![One eval, two suites, one number](images/scored-suite.svg)

The eval never changes. Only the suite under it does, which is what makes the two numbers
comparable.

## The cost

```
loss = bid changes applied more than once x spend per line item per hour
```

A read that runs twice costs latency. A bid change applied by three tools changes what you pay for
every impression until somebody notices.

## The failure

The suite as it stands. Three ways to read a floor, three to change a bid, three to stop a line
item.

In [1]:
LINE = {"line_item_id": {"type": "string"}}


def fn(name, description, props):
    """One tool. Every argument required, nothing extra allowed."""
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}}

Fourteen tools, and the names alone show where the overlap is.

In [2]:
READS = [("get_bid_floor", "the current bid floor"), ("get_floor_price", "the floor price"),
         ("get_reserve_price", "the reserve price set by the exchange"),
         ("get_win_rate", "the auction win rate"), ("get_auction_stats", "auction statistics"),
         ("get_delivery_report", "delivery and pacing figures"),
         ("get_spend_pacing", "how fast the budget is being spent")]
WRITES = [("adjust_bid", "Adjust the bid on a line item.", {"amount": {"type": "number"}}),
          ("set_bid_multiplier", "Set the bid multiplier.", {"multiplier": {"type": "number"}}),
          ("update_bid_modifier", "Update the bid modifier.", {"modifier": {"type": "number"}}),
          ("pause_campaign", "Pause a campaign.", {}), ("pause_line_item", "Pause a line item.", {}),
          ("disable_line_item", "Disable a line item.", {}),
          ("block_domain", "Block a domain for a line item.", {"domain": {"type": "string"}})]

BLOATED = [fn(name, f"Return {what} for a line item.", dict(LINE)) for name, what in READS]
BLOATED += [fn(name, description, {**LINE, **extra}) for name, description, extra in WRITES]
print(f"{len(BLOATED)} tools, {len(READS)} reads and {len(WRITES)} writes")

14 tools, 7 reads and 7 writes


Six things a trader asks for in a normal afternoon, and the one tool wired to each.

In [3]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("10-low-entropy-tool-design/03-capstone-a-scored-tool-suite")

TASKS = ["LI-4471 is bidding into a floor it cannot win. What is the floor right now?",
         "LI-4471 is spending too fast. Take its bids down 20 percent.",
         "Kill LI-4471 for now, we will come back to it tomorrow.",
         "How often are we actually winning on LI-4471?",
         "Stop us bidding on gambling-news.example entirely for LI-4471.",
         "LI-4471 is way behind its daily budget. How is the pace looking?"]

One turn, returning the calls with their arguments, because the argument is half the answer.

In [4]:
import json

SYSTEM = "You are a real time bidding operations assistant. Answer with tool calls."


def ask(tools, task):
    """One turn. Returns every call as a name and a dictionary of arguments."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=700, tools=tools, tool_choice="required",
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": task}])
    return [(call.function.name, json.loads(call.function.arguments))
            for call in (reply.choices[0].message.tool_calls or [])]

The scorer takes a name, an argument and a value, and passes only when exactly one call matched all
three.

In [5]:
def score(tools, expected, repeats=2):
    """Run every task, return the passes, the attempts and every call made."""
    passed, attempts, made = 0, 0, []
    for task, want in zip(TASKS, expected):
        for _ in range(repeats):
            calls = ask(tools, task)
            attempts += 1
            made.extend(calls)
            name, field, value = want
            ok = len(calls) == 1 and calls[0][0] == name and calls[0][1].get(field) == value
            passed += ok
    return passed, attempts, made

The bloated suite needs its own expectations, because a floor and a win rate are separate tools
there. A read carries no decisive argument, so `line_item_id` stands in.

In [6]:
BLOATED_EXPECTED = [("get_bid_floor", "line_item_id", "LI-4471"),
                    ("set_bid_multiplier", "multiplier", 0.8),
                    ("pause_line_item", "line_item_id", "LI-4471"),
                    ("get_win_rate", "line_item_id", "LI-4471"),
                    ("block_domain", "domain", "gambling-news.example"),
                    ("get_spend_pacing", "line_item_id", "LI-4471")]

bloated_passed, bloated_tried, bloated_calls = score(BLOATED, BLOATED_EXPECTED)
print(f"bloated suite: {bloated_passed}/{bloated_tried} = {bloated_passed / bloated_tried:.0%}")
print(f"backend calls: {len(bloated_calls)} for {bloated_tried} tasks")

assert bloated_passed == bloated_tried, f"{bloated_tried - bloated_passed} tasks were not done right"

bloated suite: 5/12 = 42%
backend calls: 61 for 12 tasks


AssertionError: 7 tasks were not done right

## The diagnosis

Five of twelve. Nothing exotic happened. Several tools were plausible for one sentence, so the model
asked for several.

**The read went to the wrong dial.** "What is the floor right now?" landed on `get_floor_price` both
times. Three tools carry the word floor and one is wired to the exchange.

**The write went to all three dials.** Cutting bids by a fifth produced calls to `adjust_bid`,
`set_bid_multiplier` and `update_bid_modifier`, each carrying 0.8. An amount of 0.8 is not a
multiplier of 0.8.

**Blocking a domain also stopped the line item.** `disable_line_item` and `block_domain` came back
together twice. The trader asked to block one site and would have lost all delivery.

The model read every request correctly. It could not tell which dial was the real one, because the
suite never says.

Look at the write calls on their own. This is the part that costs money.

In [7]:
from collections import Counter

BID_TOOLS = {"adjust_bid", "set_bid_multiplier", "update_bid_modifier"}
bid_calls = Counter(name for name, _ in bloated_calls if name in BID_TOOLS)
stoppers = Counter(name for name, _ in bloated_calls
                   if name in {"pause_campaign", "pause_line_item", "disable_line_item"})

print(f"bid changes asked for : {dict(bid_calls)}")
print(f"stop tools asked for  : {dict(stoppers)}")

bid changes asked for : {'adjust_bid': 17, 'set_bid_multiplier': 3, 'update_bid_modifier': 3}
stop tools asked for  : {'pause_line_item': 4, 'disable_line_item': 3, 'pause_campaign': 1}


## The fix

Four tools. The read family becomes one tool with a closed list of metrics. Three ways to change a
bid become one. Three ways to stop a line item become a state, which also makes starting it again
possible without a fourth tool.

In [8]:
METRICS = ["floor_price", "win_rate", "spend_pace", "fill_rate"]
SCOPED = [
    fn("get_auction_metric", "Return one auction metric for a line item.",
       {**LINE, "metric": {"type": "string", "enum": METRICS}}),
    fn("set_bid_multiplier", "Multiply the current bid. 1.0 leaves it unchanged.",
       {**LINE, "multiplier": {"type": "number", "minimum": 0.1, "maximum": 3.0}}),
    fn("set_line_item_state", "Set the delivery state of a line item.",
       {**LINE, "state": {"type": "string", "enum": ["active", "paused", "stopped"]}}),
    fn("block_placement", "Stop a line item bidding on one domain.",
       {**LINE, "domain": {"type": "string"}}),
]
print(f"{len(BLOATED)} tools became {len(SCOPED)}, and every choice is now a checked value")

14 tools became 4, and every choice is now a checked value


Same eval, same tasks, same scorer. Only the suite underneath has changed.

In [9]:
SCOPED_EXPECTED = [("get_auction_metric", "metric", "floor_price"),
                   ("set_bid_multiplier", "multiplier", 0.8),
                   ("set_line_item_state", "state", "paused"),
                   ("get_auction_metric", "metric", "win_rate"),
                   ("block_placement", "domain", "gambling-news.example"),
                   ("get_auction_metric", "metric", "spend_pace")]

scoped_passed, scoped_tried, scoped_calls = score(SCOPED, SCOPED_EXPECTED)

print(f"bloated: {bloated_passed:2}/{bloated_tried} = {bloated_passed / bloated_tried:3.0%}"
      f"   {len(bloated_calls):3} backend calls")
print(f"scoped : {scoped_passed:2}/{scoped_tried} = {scoped_passed / scoped_tried:3.0%}"
      f"   {len(scoped_calls):3} backend calls")
print(f"change : {(scoped_passed - bloated_passed) / bloated_tried:+.0%} correct, "
      f"{len(scoped_calls) - len(bloated_calls):+} calls")

bloated:  5/12 = 42%    61 backend calls
scoped : 12/12 = 100%    12 backend calls
change : +58% correct, -49 calls


## The gate

The eval is the gate. It runs against recorded answers, so it costs nothing and fails the build when
a fifteenth tool brings the overlap back.

In [10]:
FLOOR = 1.0


def test_the_suite_scores_at_the_floor():
    passed, tried, _ = score(SCOPED, SCOPED_EXPECTED)
    rate = passed / tried
    assert rate >= FLOOR, f"the suite scores {rate:.0%}, the floor is {FLOOR:.0%}"


test_the_suite_scores_at_the_floor()
print(f"gate holds: {len(SCOPED)} tools, {len(TASKS)} tasks, scoring at or above {FLOOR:.0%}")

gate holds: 4 tools, 6 tasks, scoring at or above 100%


Put `adjust_bid` back beside `set_bid_multiplier` and the score drops below the floor, so the build
fails before the suite ships.

### Enterprise exploration

- The gate replays recordings. What tells you they have gone stale, and how often would you spend
  real money to find out?
- Collapsing three write tools into one changes an audit trail finance depends on. What is the
  compliance cost of losing the old names, and what would you keep?
- The floor is 100 percent on six tasks. At sixty tasks it will not be. What is the right floor, and
  what happens the day a model upgrade drops the score four points?

### Key takeaways

- A tool suite is a design you can measure. Score it before and after, with the eval held still.
- Overlapping writes are worse than overlapping reads. One turn applied the same cut three ways.
- Closed lists move a decision out of a tool name and into an argument that gets checked.
- An eval running on recordings is cheap enough to sit in the build, the only place it stops a
  regression.